# Fake News Classification: DistilBERT vs Traditional ML Baselines
## NLP Assignment — 662253 — 2024/2025

**Research Question:** Can a fine-tuned DistilBERT model significantly outperform traditional machine learning baselines for fake news classification, and what linguistic features most contribute to misinformation detection?

**Dataset:** LIAR (Wang, 2017) — binary classification (true vs false)

**Models:**
1. Majority Class Baseline
2. TF-IDF + Logistic Regression
3. TF-IDF + Naive Bayes
4. Fine-tuned DistilBERT (main system)

## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import warnings
import os
import re
import time

warnings.filterwarnings('ignore')

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

# PyTorch + HuggingFace Transformers
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device — use MPS on Apple Silicon, CUDA on GPU, else CPU
if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

print(f'Using device: {DEVICE}')

## 2. Data Loading and Exploration

In [ ]:
import os

# Set working directory to the notebook location so relative paths work
os.chdir(os.path.dirname(os.path.abspath('fake_news_classification.ipynb')))

# Load the LIAR dataset from locally downloaded TSV files (Wang, 2017)
# Dataset: https://www.cs.ucsb.edu/~william/data/liar_dataset.zip
DATA_DIR = 'liar_data'

# TSV column schema — no header row in files
COLS = [
    'id', 'label', 'statement', 'subject', 'speaker',
    'job_title', 'state', 'party',
    'barely_true_cnt', 'false_cnt', 'half_true_cnt', 'mostly_true_cnt', 'pants_fire_cnt',
    'context'
]

LABEL_NAMES_6 = ['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
LABEL2ID      = {name: idx for idx, name in enumerate(LABEL_NAMES_6)}

def load_split(filename):
    import pandas as pd
    df = pd.read_csv(
        os.path.join(DATA_DIR, filename),
        sep='\t', header=None, names=COLS, quoting=3
    )
    return df

raw_train = load_split('train.tsv')
raw_val   = load_split('valid.tsv')
raw_test  = load_split('test.tsv')

print(f'Loaded — Train: {len(raw_train):,} | Val: {len(raw_val):,} | Test: {len(raw_test):,}')
print(f'\nLabel distribution (train):\n{raw_train["label"].value_counts()}')

In [ ]:
# Binarise labels: TRUE = {mostly-true, true} → 1 ; FALSE = {false, half-true, barely-true, pants-fire} → 0
# Rationale: half-true statements contain a mix of accurate and inaccurate elements and are grouped
# with false claims following Wang (2017)'s binary grouping convention.
TRUE_LABELS  = {'mostly-true', 'true'}
FALSE_LABELS = {'false', 'half-true', 'barely-true', 'pants-fire'}

def binarise(df):
    df = df.copy()
    df['binary_label'] = df['label'].apply(lambda x: 1 if x in TRUE_LABELS else 0)
    return df

def make_split(raw):
    df = binarise(raw)[['statement', 'binary_label', 'speaker', 'subject', 'label']]
    df = df.rename(columns={'statement': 'text', 'binary_label': 'label', 'label': 'label_orig'})
    return df

train_df = make_split(raw_train)
val_df   = make_split(raw_val)
test_df  = make_split(raw_test)

print('Binary label distribution (train):')
print(train_df['label'].value_counts().rename({0: 'FALSE (0)', 1: 'TRUE (1)'}))
print(f'\nClass balance: {train_df["label"].mean():.1%} TRUE statements')
print('\nSample rows:')
train_df[['text', 'label', 'label_orig']].head(3)

In [ ]:
y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

print(f'Split sizes — Train: {len(y_train):,} | Validation: {len(y_val):,} | Test: {len(y_test):,}')
print(f'\nTrain class balance: {y_train.mean():.1%} TRUE | {1 - y_train.mean():.1%} FALSE')

In [ ]:
# --- Figure 1: Class distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Binary distribution
counts = train_df['label'].value_counts().sort_index()
axes[0].bar(['FALSE (0)', 'TRUE (1)'], counts.values, color=['#e74c3c', '#2ecc71'], edgecolor='black')
axes[0].set_title('Binary Label Distribution (Train)', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Original 6-class distribution
raw_counts = raw_train['label'].value_counts()
raw_counts = raw_counts.reindex(LABEL_NAMES_6, fill_value=0)
axes[1].bar(LABEL_NAMES_6, raw_counts.values, color=sns.color_palette('husl', 6), edgecolor='black')
axes[1].set_title('Original 6-Class Distribution (Train)', fontweight='bold')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)
for i, v in enumerate(raw_counts.values):
    axes[1].text(i, v + 10, str(v), ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.savefig('fig1_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig1_class_distribution.png')

In [ ]:
# --- Figure 2: Statement length distribution ---
train_df['text_len'] = train_df['text'].apply(lambda x: len(x.split()))

fig, ax = plt.subplots(figsize=(9, 4))
for label, colour, name in [(0, '#e74c3c', 'FALSE'), (1, '#2ecc71', 'TRUE')]:
    ax.hist(train_df[train_df['label'] == label]['text_len'],
            bins=40, alpha=0.6, color=colour, label=name, edgecolor='none')
ax.set_xlabel('Statement Length (words)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Statement Lengths by Class', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig2_length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mean length — FALSE: {train_df[train_df['label']==0]['text_len'].mean():.1f} words | "
      f"TRUE: {train_df[train_df['label']==1]['text_len'].mean():.1f} words")

## 3. Text Preprocessing

In [ ]:
import re

def clean_text(text: str) -> str:
    """Lightweight preprocessing for TF-IDF models."""
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s']", ' ', text)  # keep apostrophes
    text = re.sub(r'\s+', ' ', text).strip()
    return text

X_train = train_df['text'].apply(clean_text)
X_val   = val_df['text'].apply(clean_text)
X_test  = test_df['text'].apply(clean_text)

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

print('Preprocessing complete.')
print(f'Example: "{train_df["text"].iloc[0]}"')
print(f'Cleaned: "{X_train.iloc[0]}"')

## 4. Baseline 1 — Majority Class Classifier

In [ ]:
majority = DummyClassifier(strategy='most_frequent', random_state=SEED)
majority.fit(X_train, y_train)
y_pred_majority = majority.predict(X_test)

acc_majority = accuracy_score(y_test, y_pred_majority)
f1_majority  = f1_score(y_test, y_pred_majority, average='macro', zero_division=0)

print('=== Majority Class Baseline ===')
print(f'Accuracy:  {acc_majority:.4f}')
print(f'Macro F1:  {f1_majority:.4f}')
print(classification_report(y_test, y_pred_majority, target_names=['FALSE', 'TRUE'], zero_division=0))

## 5. Baseline 2 — TF-IDF + Logistic Regression

In [ ]:
lr_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        solver='lbfgs',
        random_state=SEED
    ))
])

lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr, average='macro')

print('=== TF-IDF + Logistic Regression ===')
print(f'Accuracy:  {acc_lr:.4f}')
print(f'Macro F1:  {f1_lr:.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['FALSE', 'TRUE']))

In [ ]:
# --- Figure 3: Top predictive n-grams from Logistic Regression ---
tfidf_vocab = lr_pipe.named_steps['tfidf'].get_feature_names_out()
lr_coefs    = lr_pipe.named_steps['clf'].coef_[0]

top_n = 15
top_false_idx = np.argsort(lr_coefs)[:top_n]     # most negative → FALSE
top_true_idx  = np.argsort(lr_coefs)[-top_n:][::-1]  # most positive → TRUE

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, idx, colour, title in [
    (axes[0], top_false_idx, '#e74c3c', 'Top 15 FALSE-predictive n-grams'),
    (axes[1], top_true_idx,  '#2ecc71', 'Top 15 TRUE-predictive n-grams'),
]:
    features = tfidf_vocab[idx]
    coefs    = lr_coefs[idx]
    ax.barh(range(top_n), np.abs(coefs), color=colour, edgecolor='black', alpha=0.85)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(features, fontsize=10)
    ax.invert_yaxis()
    ax.set_xlabel('|Coefficient|')
    ax.set_title(title, fontweight='bold')

plt.suptitle('Logistic Regression Feature Importance (TF-IDF Coefficients)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_lr_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig3_lr_feature_importance.png')

## 6. Baseline 3 — TF-IDF + Naive Bayes

In [ ]:
nb_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        sublinear_tf=False,   # NB works better with raw TF
        min_df=2
    )),
    ('clf', MultinomialNB(alpha=0.1))
])

nb_pipe.fit(X_train, y_train)
y_pred_nb = nb_pipe.predict(X_test)

acc_nb = accuracy_score(y_test, y_pred_nb)
f1_nb  = f1_score(y_test, y_pred_nb, average='macro')

print('=== TF-IDF + Naive Bayes ===')
print(f'Accuracy:  {acc_nb:.4f}')
print(f'Macro F1:  {f1_nb:.4f}')
print(classification_report(y_test, y_pred_nb, target_names=['FALSE', 'TRUE']))

## 7. Main System — Fine-tuned DistilBERT

In [ ]:
MODEL_NAME  = 'distilbert-base-uncased'
MAX_LEN     = 128
BATCH_SIZE  = 32
EPOCHS      = 4
LR          = 2e-5
WARMUP_FRAC = 0.1

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print('Tokenizer loaded.')

In [ ]:
class LiarDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=max_len,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

train_dataset = LiarDataset(train_df['text'].tolist(), y_train, tokenizer, MAX_LEN)
val_dataset   = LiarDataset(val_df['text'].tolist(),   y_val,   tokenizer, MAX_LEN)
test_dataset  = LiarDataset(test_df['text'].tolist(),  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}')

In [ ]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to(DEVICE)

total_steps   = len(train_loader) * EPOCHS
warmup_steps  = int(WARMUP_FRAC * total_steps)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Total training steps: {total_steps} | Warmup steps: {warmup_steps}')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = outputs.logits.argmax(dim=-1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            total_loss += outputs.loss.item()
            preds = outputs.logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1, np.array(all_preds), np.array(all_labels)

print('Training functions defined.')

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'val_f1': []}
best_val_f1 = 0.0
best_model_path = 'best_distilbert.pt'

print(f'Training DistilBERT for {EPOCHS} epochs on {DEVICE}...\n')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, val_loader, DEVICE)
    elapsed = time.time() - t0

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)

    marker = '  *** best ***' if val_f1 > best_val_f1 else ''
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), best_model_path)

    print(f'Epoch {epoch}/{EPOCHS} ({elapsed:.0f}s) | '
          f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}  Val F1: {val_f1:.4f}{marker}')

print(f'\nBest validation Macro F1: {best_val_f1:.4f}')

In [ ]:
# --- Figure 4: Training curves ---
epochs_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='#3498db')
axes[0].plot(epochs_range, history['val_loss'],   's--', label='Val Loss',   color='#e74c3c')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training and Validation Loss', fontweight='bold')
axes[0].legend()
axes[0].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

axes[1].plot(epochs_range, history['train_acc'], 'o-', label='Train Acc', color='#3498db')
axes[1].plot(epochs_range, history['val_acc'],   's--', label='Val Acc',   color='#e74c3c')
axes[1].plot(epochs_range, history['val_f1'],    '^:', label='Val Macro F1', color='#9b59b6')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Training and Validation Accuracy / F1', fontweight='bold')
axes[1].legend()
axes[1].xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('fig4_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig4_training_curves.png')

In [ ]:
# Evaluate best model on test set
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
_, acc_bert, f1_bert, y_pred_bert, _ = evaluate(model, test_loader, DEVICE)

print('=== Fine-tuned DistilBERT (Test Set) ===')
print(f'Accuracy:  {acc_bert:.4f}')
print(f'Macro F1:  {f1_bert:.4f}')
print(classification_report(y_test, y_pred_bert, target_names=['FALSE', 'TRUE']))

## 8. Results and Comparison

In [ ]:
# --- Table 1: Summary results ---
results = pd.DataFrame([
    {'Model': 'Majority Class Baseline',        'Accuracy': acc_majority, 'Macro F1': f1_majority,
     'Precision (FALSE)': precision_score(y_test, y_pred_majority, pos_label=0, zero_division=0),
     'Recall (FALSE)':    recall_score(y_test, y_pred_majority, pos_label=0, zero_division=0),
     'Precision (TRUE)':  precision_score(y_test, y_pred_majority, pos_label=1, zero_division=0),
     'Recall (TRUE)':     recall_score(y_test, y_pred_majority, pos_label=1, zero_division=0)},
    {'Model': 'TF-IDF + Naive Bayes',           'Accuracy': acc_nb, 'Macro F1': f1_nb,
     'Precision (FALSE)': precision_score(y_test, y_pred_nb, pos_label=0),
     'Recall (FALSE)':    recall_score(y_test, y_pred_nb, pos_label=0),
     'Precision (TRUE)':  precision_score(y_test, y_pred_nb, pos_label=1),
     'Recall (TRUE)':     recall_score(y_test, y_pred_nb, pos_label=1)},
    {'Model': 'TF-IDF + Logistic Regression',   'Accuracy': acc_lr, 'Macro F1': f1_lr,
     'Precision (FALSE)': precision_score(y_test, y_pred_lr, pos_label=0),
     'Recall (FALSE)':    recall_score(y_test, y_pred_lr, pos_label=0),
     'Precision (TRUE)':  precision_score(y_test, y_pred_lr, pos_label=1),
     'Recall (TRUE)':     recall_score(y_test, y_pred_lr, pos_label=1)},
    {'Model': 'Fine-tuned DistilBERT',           'Accuracy': acc_bert, 'Macro F1': f1_bert,
     'Precision (FALSE)': precision_score(y_test, y_pred_bert, pos_label=0),
     'Recall (FALSE)':    recall_score(y_test, y_pred_bert, pos_label=0),
     'Precision (TRUE)':  precision_score(y_test, y_pred_bert, pos_label=1),
     'Recall (TRUE)':     recall_score(y_test, y_pred_bert, pos_label=1)},
])

float_cols = [c for c in results.columns if c != 'Model']
print('\n=== Table 1: Test Set Results ===')
print(results.to_string(index=False, float_format='{:.4f}'.format))
results[float_cols] = results[float_cols].round(4)

In [ ]:
# --- Figure 5: Model comparison bar chart ---
fig, ax = plt.subplots(figsize=(10, 5))
x       = np.arange(len(results))
width   = 0.35

bars1 = ax.bar(x - width/2, results['Accuracy'], width, label='Accuracy', color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, results['Macro F1'], width, label='Macro F1', color='#e67e22', edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(results['Model'], rotation=15, ha='right', fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Model Comparison: Accuracy and Macro F1 on Test Set', fontweight='bold')
ax.legend()

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('fig5_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig5_model_comparison.png')

In [ ]:
# --- Figure 6: Confusion matrices for all four models ---
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

models_and_preds = [
    ('Majority Baseline', y_pred_majority),
    ('TF-IDF + Naive Bayes', y_pred_nb),
    ('TF-IDF + Logistic Regression', y_pred_lr),
    ('Fine-tuned DistilBERT', y_pred_bert),
]

for ax, (name, y_pred) in zip(axes.flatten(), models_and_preds):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['FALSE', 'TRUE'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')

plt.suptitle('Confusion Matrices — All Models (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('fig6_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig6_confusion_matrices.png')

## 9. Additional Analysis — DistilBERT Embeddings as Features (Ablation)

In [ ]:
# Extract CLS-token embeddings from the frozen DistilBERT and feed into a Logistic Regression.
# This isolates the contribution of contextual representations vs. end-to-end fine-tuning.

frozen_model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
frozen_model = frozen_model.to(DEVICE)
frozen_model.eval()

def extract_embeddings(loader, model, device):
    all_emb = []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            # Access the underlying DistilBert model for hidden states
            outputs = model.distilbert(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            cls_emb = outputs.last_hidden_state[:, 0, :]  # [CLS] token
            all_emb.append(cls_emb.cpu().numpy())
    return np.vstack(all_emb)

print('Extracting embeddings from frozen DistilBERT (this may take ~1 minute)...')
train_emb = extract_embeddings(train_loader, frozen_model, DEVICE)
test_emb  = extract_embeddings(test_loader,  frozen_model, DEVICE)
print(f'Embedding shape: {train_emb.shape}')

In [ ]:
scaler           = StandardScaler()
train_emb_scaled = scaler.fit_transform(train_emb)
test_emb_scaled  = scaler.transform(test_emb)

lr_emb = LogisticRegression(max_iter=1000, C=1.0, random_state=SEED)
lr_emb.fit(train_emb_scaled, y_train)
y_pred_emb = lr_emb.predict(test_emb_scaled)

acc_emb = accuracy_score(y_test, y_pred_emb)
f1_emb  = f1_score(y_test, y_pred_emb, average='macro')

print('=== DistilBERT Embeddings + Logistic Regression (Ablation) ===')
print(f'Accuracy:  {acc_emb:.4f}')
print(f'Macro F1:  {f1_emb:.4f}')
print(classification_report(y_test, y_pred_emb, target_names=['FALSE', 'TRUE']))

In [ ]:
# --- Figure 7: Full ablation comparison ---
ablation_data = {
    'Majority\nBaseline': (acc_majority, f1_majority),
    'TF-IDF +\nNaive Bayes': (acc_nb, f1_nb),
    'TF-IDF +\nLog. Reg.': (acc_lr, f1_lr),
    'DistilBERT\nEmbeds + LR': (acc_emb, f1_emb),
    'Fine-tuned\nDistilBERT': (acc_bert, f1_bert),
}

names  = list(ablation_data.keys())
accs   = [v[0] for v in ablation_data.values()]
f1s    = [v[1] for v in ablation_data.values()]
x      = np.arange(len(names))

fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - 0.2, accs, 0.35, label='Accuracy', color='#3498db', edgecolor='black')
b2 = ax.bar(x + 0.2, f1s,  0.35, label='Macro F1', color='#e67e22', edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title('Ablation Study: Contribution of DistilBERT Representations vs. Fine-Tuning', fontweight='bold')
ax.legend()
ax.axvline(x=2.5, color='grey', linestyle='--', linewidth=1.5, label='transformer boundary')
ax.text(2.6, 0.95, 'transformer\nmodels →', fontsize=9, color='grey')
for b in list(b1) + list(b2):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('fig7_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: fig7_ablation.png')

## 10. Final Summary

In [ ]:
full_results = pd.DataFrame([
    {'Model': 'Majority Class Baseline',             'Accuracy': acc_majority, 'Macro F1': f1_majority},
    {'Model': 'TF-IDF + Naive Bayes',                'Accuracy': acc_nb,       'Macro F1': f1_nb},
    {'Model': 'TF-IDF + Logistic Regression',        'Accuracy': acc_lr,       'Macro F1': f1_lr},
    {'Model': 'DistilBERT Embeddings + Log. Reg.',   'Accuracy': acc_emb,      'Macro F1': f1_emb},
    {'Model': 'Fine-tuned DistilBERT (main system)', 'Accuracy': acc_bert,     'Macro F1': f1_bert},
])

print('\n=== FINAL RESULTS TABLE ===')
print(full_results.to_string(index=False, float_format='{:.4f}'.format))

best_row = full_results.loc[full_results['Macro F1'].idxmax()]
print(f'\nBest model by Macro F1: {best_row["Model"]}  (F1 = {best_row["Macro F1"]:.4f})')

improvement = (f1_bert - f1_majority) / f1_majority * 100 if f1_majority > 0 else float('inf')
print(f'DistilBERT Macro F1 improvement over majority baseline: +{improvement:.1f}%')

## Hyperparameter Summary

| Component | Setting |
|---|---|
| DistilBERT pretrained model | `distilbert-base-uncased` |
| Max sequence length | 128 tokens |
| Batch size | 32 |
| Epochs | 4 |
| Learning rate | 2e-5 |
| LR schedule | Linear with 10% warmup |
| Weight decay | 0.01 |
| Gradient clipping | max norm = 1.0 |
| TF-IDF features | 50,000 unigrams + bigrams |
| Logistic Regression C | 1.0 |
| Naive Bayes alpha | 0.1 |